# Day 29 practice — Building an API

**Focus:** Flask CRUD endpoints, JSON request/response, validation, and status codes.

**Review:** [lesson notes](29_building_API.md).

**Requirements:** Flask required. Install it into the notebook kernel with %pip install flask if needed. Use app.test_client(); no external server is needed.

Work through the exercises in order. Edit the TODO cells and run them yourself. The notebook contains prompts and starting data, not worked solutions. When a task uses repository data, run the kernel from either the repository root or this lesson folder.

**Example solutions:** Each solution is collapsed by default. Try the exercise first, then expand it if you want to compare approaches. Copy code into the TODO cell only when needed.

**Flask note:** After handling a request, Flask does not allow new routes on that app. Create a fresh app and test client when you add routes in later exercises.

## Exercise 1: Initialize the API

Create a Flask app and an in-memory list of two student dictionaries with numeric id, name, and country. Add GET /api/students to return JSON and status 200; test it with app.test_client().

In [ ]:
# TODO: Write your solution here.


<details>
<summary>Show example solution</summary>

```python
from flask import Flask, jsonify, request

students = [
    {'id': 1, 'name': 'Ana', 'country': 'Finland'},
    {'id': 2, 'name': 'Ben', 'country': 'Sweden'},
]
routes = []

def register(rule, endpoint, view, methods):
    routes.append((rule, endpoint, view, methods))

def fresh_client():
    app = Flask(__name__)
    for rule, endpoint, view, methods in routes:
        app.add_url_rule(rule, endpoint=endpoint, view_func=view, methods=methods)
    return app.test_client()

def list_students():
    return jsonify(students), 200

register('/api/students', 'list_students', list_students, ['GET'])
client = fresh_client()
response = client.get('/api/students')
print(response.status_code, response.get_json())
```

</details>

## Exercise 2: Get one student

Add GET /api/students/<int:student_id>. Return the matching student, or a JSON error with status 404. Test both an existing and a missing ID.

In [ ]:
# TODO: Write your solution here.


<details>
<summary>Show example solution</summary>

```python
def get_student(student_id):
    student = next((item for item in students if item['id'] == student_id), None)
    if student is None:
        return jsonify({'error': 'Student not found'}), 404
    return jsonify(student), 200

register('/api/students/<int:student_id>', 'get_student', get_student, ['GET'])
client = fresh_client()
print(client.get('/api/students/1').status_code)
print(client.get('/api/students/999').status_code)
```

</details>

## Exercise 3: Create a student

Add POST /api/students. Require nonempty name and country in JSON; return status 400 on invalid input, or append a new student with a unique ID and return status 201. Test both cases.

In [ ]:
# TODO: Write your solution here.


<details>
<summary>Show example solution</summary>

```python
def create_student():
    data = request.get_json(silent=True) or {}
    name, country = data.get('name'), data.get('country')
    if not isinstance(name, str) or not name.strip() or not isinstance(country, str) or not country.strip():
        return jsonify({'error': 'Name and country are required'}), 400
    student = {'id': max((item['id'] for item in students), default=0) + 1,
               'name': name.strip(), 'country': country.strip()}
    students.append(student)
    return jsonify(student), 201

register('/api/students', 'create_student', create_student, ['POST'])
client = fresh_client()
print(client.post('/api/students', json={'name': 'Cara', 'country': 'Norway'}).status_code)
print(client.post('/api/students', json={'name': ''}).status_code)
```

</details>

## Exercise 4: Update a student

Add PUT /api/students/<int:student_id> to replace or update the supplied fields. Return 404 when the ID is absent. Verify the changed record with GET.

In [ ]:
# TODO: Write your solution here.


<details>
<summary>Show example solution</summary>

```python
def update_student(student_id):
    student = next((item for item in students if item['id'] == student_id), None)
    if student is None:
        return jsonify({'error': 'Student not found'}), 404
    data = request.get_json(silent=True)
    if not isinstance(data, dict):
        return jsonify({'error': 'JSON object required'}), 400
    for key in ('name', 'country'):
        if key in data:
            if not isinstance(data[key], str) or not data[key].strip():
                return jsonify({'error': f'Invalid {key}'}), 400
            student[key] = data[key].strip()
    return jsonify(student), 200

register('/api/students/<int:student_id>', 'update_student', update_student, ['PUT'])
client = fresh_client()
print(client.put('/api/students/1', json={'country': 'Norway'}).status_code)
print(client.get('/api/students/1').get_json())
```

</details>

## Exercise 5: Delete a student

Add DELETE /api/students/<int:student_id>. Remove the record and return a suitable success status; return 404 for a missing ID. Verify a second GET no longer finds it.

In [ ]:
# TODO: Write your solution here.


<details>
<summary>Show example solution</summary>

```python
def delete_student(student_id):
    for index, student in enumerate(students):
        if student['id'] == student_id:
            students.pop(index)
            return jsonify({'deleted_id': student_id}), 200
    return jsonify({'error': 'Student not found'}), 404

register('/api/students/<int:student_id>', 'delete_student', delete_student, ['DELETE'])
client = fresh_client()
print(client.delete('/api/students/2').status_code)
print(client.get('/api/students/2').status_code)
print(client.delete('/api/students/2').status_code)
```

</details>

## Reflection

Which exercise was hardest? What would you try again without looking at the lesson notes?